## Passo 1: Carga dos dados no Banco de Dados
Para cumprir o primeiro requisito do teste, vamos ler os arquivos originais em formato CSV e carregá-los diretamente em um banco de dados local usando o **SQLite**.

In [ ]:
import pandas as pd
import sqlite3

# 1. Ler os arquivos CSV originais (
df1 = pd.read_csv('artigos_fi1.csv')
df2 = pd.read_csv('artigos_fi2.csv')

# 2. Criar e conectar ao banco de dados SQLite (ele cria o arquivo .db na hora)
conexao = sqlite3.connect('banco_agora_sabemos.db')

# 3. Carregar as duas bases para dentro do banco de dados
df1.to_sql('tabela_artigos_fi1', conexao, if_exists='replace', index=False)
df2.to_sql('tabela_artigos_fi2', conexao, if_exists='replace', index=False)

# 4. Fechar a conexão com o banco
conexao.close()

print("✅ Requisito 1 concluído: As duas bases originais foram carregadas no banco SQLite com sucesso!")

## Passo 2: Unificação e Limpeza Básica dos Dados
Nesta etapa, realizamos a padronização dos nomes das colunas para evitar erros de case-sensitive (letras maiúsculas e minúsculas). Em seguida, unimos as duas bases de dados e tratamos os valores nulos nas colunas principais (`qualis` e `sjr`), além de remover possíveis linhas duplicadas.

In [ ]:
# 1. Padronizar nomes das colunas (tudo em minúsculo para evitar conflitos)
df1.columns = df1.columns.str.lower()
df2.columns = df2.columns.str.lower()

# 2. Unificar as duas bases em um único DataFrame
df_full = pd.concat([df1, df2], ignore_index=True)

# 3. Tratar valores nulos (Limpeza Básica)
# Para o QUALIS: Se não tiver nota, preenchemos com 'SEM QUALIS' e deixamos tudo maiúsculo
if 'qualis' in df_full.columns:
    df_full['qualis'] = df_full['qualis'].fillna('SEM QUALIS').str.upper().str.strip()

# Para o SJR: Se a revista não tiver pontuação, preenchemos com 0
if 'sjr' in df_full.columns:
    df_full['sjr'] = df_full['sjr'].fillna(0)

# 4. Remover linhas que sejam 100% duplicadas (se houver)
df_full = df_full.drop_duplicates()

# Mensagem de sucesso para confirmar
print(f"✅ Passo 2 concluído: Base unificada e limpa com sucesso!")
print(f"📊 Total de linhas prontas para análise: {len(df_full)}")

## Passo 3: Criação da Tabela Unificada para Análise Cruzada
Para cumprir o requisito de consolidação, vamos salvar o DataFrame unificado e limpo (`df_full`) de volta no banco de dados SQLite. Esta tabela consolidada (`tabela_unificada_analise`) servirá como fonte única da verdade para as análises e para a conexão com a ferramenta de BI.

In [ ]:
import sqlite3

# 1. Conectar novamente ao banco de dados que criamos no Passo 1
conexao = sqlite3.connect('banco_agora_sabemos.db')

# 2. Salvar o DataFrame limpo e unificado como uma nova tabela no banco
df_full.to_sql('tabela_unificada_analise', conexao, if_exists='replace', index=False)

# 3. Teste rápido (Query SQL): Ler a tabela recém-criada só para provar que deu certo
query = "SELECT * FROM tabela_unificada_analise LIMIT 5"
teste_leitura = pd.read_sql(query, conexao)

# 4. Fechar a conexão
conexao.close()

# Exibir resultado
print("✅ Passo 3 concluído: Tabela unificada 'tabela_unificada_analise' criada com sucesso no banco de dados!")
display(teste_leitura)

### 🛠️ Tratamento Específico: Coluna SJR (Garantia de Qualidade - QA)
Durante a auditoria dos dados, identifiquei uma inconsistência na coluna `sjr`. Os valores decimais originais estavam separados por vírgula (ex: 23,962). Isso faz com que os sistemas interpretem a métrica de forma incorreta (como texto, booleano ou arredondando para inteiro).

Para garantir a precisão matemática no Power BI — especialmente para o Gráfico de Dispersão e o Ranking Top 5 — apliquei uma etapa de conversão: substituímos as vírgulas por pontos (padrão internacional) e forçamos a tipagem para decimal numérico (`float`).

In [ ]:
# 1. Transformar em texto, trocar vírgula por ponto
df_full['sjr'] = df_full['sjr'].astype(str).str.replace(',', '.')

# 2. Converter de volta para número decimal (float) forçando erros a ficarem vazios (NaN)
df_full['sjr'] = pd.to_numeric(df_full['sjr'], errors='coerce')

# Visualizar se deu certo
print(df_full[['sjr']].head(10))

## Passo 4: Exportação para o Power BI
Com a base unificada e limpa consolidada, a última etapa do nosso ETL é exportar esses dados para um arquivo `.csv`. Este arquivo será armazenado na nuvem (OneDrive) para alimentar o nosso dashboard no Power BI, garantindo que a visualização consuma apenas dados tratados.

In [ ]:
nome_arquivo = 'base_limpa_powerbi.csv'
df_full.to_csv(nome_arquivo, index=False)